In [1]:
# ============================================
# IMPORT LIBRARIES
# ============================================

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10

2026-05-09 17:17:07.819619: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778347028.042445      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778347028.106703      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778347028.639995      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778347028.640037      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778347028.640041      57 computation_placer.cc:177] computation placer alr

In [2]:
# ============================================
# LOAD CIFAR-10 DATASET (TRAINING)
# ============================================

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print("CIFAR Train Shape:", x_train.shape)
print("CIFAR Test Shape :", x_test.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 693s 4us/step
CIFAR Train Shape: (50000, 32, 32, 3)
CIFAR Test Shape : (10000, 32, 32, 3)


In [3]:
# ============================================
# NORMALIZE CIFAR DATA
# ============================================

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

y_train = y_train.flatten()
y_test = y_test.flatten()

In [4]:
# ============================================
# LOAD CINIC-10 TEST DATASET
# ============================================

cinic_test = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/input/datasets/mengcius/cinic10/test',
    image_size=(32,32),
    batch_size=64,
    shuffle=False
)

Found 90000 files belonging to 10 classes.


I0000 00:00:1778347876.498323      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778347876.504905      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [5]:
# ============================================
# NORMALIZE CINIC DATA
# ============================================

cinic_test = cinic_test.map(
    lambda x, y: (tf.cast(x, tf.float32) / 255.0, y)
)

In [6]:
# ============================================
# DATA AUGMENTATION
# ============================================

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

In [7]:
# ============================================
# XCEPTION BLOCK
# ============================================

def xception_block(x, filters):

    residual = layers.Conv2D(
        filters,
        (1,1),
        strides=2,
        padding='same'
    )(x)

    x = layers.SeparableConv2D(
        filters,
        (3,3),
        padding='same',
        activation='relu'
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.SeparableConv2D(
        filters,
        (3,3),
        padding='same',
        activation='relu'
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.MaxPooling2D(
        (3,3),
        strides=2,
        padding='same'
    )(x)

    x = layers.Add()([x, residual])

    return x

In [8]:
# ============================================
# BUILD XCEPTION MODEL
# ============================================

def build_xception(input_shape=(32,32,3), num_classes=10):

    inputs = layers.Input(shape=input_shape)

    x = data_augmentation(inputs)

    x = layers.Conv2D(
        32,
        (3,3),
        padding='same',
        activation='relu'
    )(x)

    x = xception_block(x, 64)

    x = xception_block(x, 128)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(128, activation='relu')(x)

    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(
        num_classes,
        activation='softmax'
    )(x)

    model = models.Model(inputs, outputs)

    return model

In [9]:
# ============================================
# CREATE MODEL
# ============================================

model = build_xception()

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 32, 32, 3) │          0 │ input_layer[0][0] │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 32, 32,    │        896 │ sequential[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d    │ (None, 32, 32,    │      2,400 │ conv2d[0][0]      │
│ (SeparableConv2D)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 32, 32,    │        256 │ separable_conv2d… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_1  │ (None, 32, 32,    │      4,736 │ batch_normalizat… │
│ (SeparableConv2D)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ separable_conv2d… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 16, 16,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 16, 16,    │      2,112 │ conv2d[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 16, 16,    │          0 │ max_pooling2d[0]… │
│                     │ 64)               │            │ conv2d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_2  │ (None, 16, 16,    │      8,896 │ add[0][0]         │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        512 │ separable_conv2d… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_3  │ (None, 16, 16,    │     17,664 │ batch_normalizat… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        512 │ separable_conv2d… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 8, 8, 128) │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 8, 8, 128) │      8,320 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 8, 8, 128) │          0 │ max_pooling2d_1[… │
│                     │                   │            │ conv2d_2[0][0]  

 Total params: 64,362 (251.41 KB)

 Trainable params: 63,594 (248.41 KB)

 Non-trainable params: 768 (3.00 KB)

In [10]:
# ============================================
# COMPILE MODEL
# ============================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [11]:
# ============================================
# TRAIN ON CIFAR-10
# ============================================

history = model.fit(
    x_train,
    y_train,
    batch_size=64,
    epochs=40,
    validation_data=(x_test, y_test)
)

Epoch 1/40


I0000 00:00:1778347974.910136     147 cuda_dnn.cc:529] Loaded cuDNN version 91002


782/782 ━━━━━━━━━━━━━━━━━━━━ 28s 25ms/step - accuracy: 0.3464 - loss: 1.7929 - val_accuracy: 0.4351 - val_loss: 1.5376
Epoch 2/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - accuracy: 0.5270 - loss: 1.3161 - val_accuracy: 0.4858 - val_loss: 1.5267
Epoch 3/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - accuracy: 0.5810 - loss: 1.1789 - val_accuracy: 0.5463 - val_loss: 1.3689
Epoch 4/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - accuracy: 0.6093 - loss: 1.0929 - val_accuracy: 0.5925 - val_loss: 1.1941
Epoch 5/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - accuracy: 0.6375 - loss: 1.0329 - val_accuracy: 0.6159 - val_loss: 1.0855
Epoch 6/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - accuracy: 0.6560 - loss: 0.9783 - val_accuracy: 0.6105 - val_loss: 1.1219
Epoch 7/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - accuracy: 0.6690 - loss: 0.9499 - val_accuracy: 0.6653 - val_loss: 1.0070
Epoch 8/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - accuracy: 0.6811 - loss: 0.9129 - val_accurac

In [12]:
# ============================================
# TEST ON CINIC-10
# ============================================

test_loss, test_acc = model.evaluate(cinic_test)

print("CIFAR → CINIC Xception Accuracy:", test_acc)

1407/1407 ━━━━━━━━━━━━━━━━━━━━ 173s 122ms/step - accuracy: 0.6576 - loss: 1.2521
CIFAR → CINIC Xception Accuracy: 0.5906000137329102


In [13]:
# ============================================
# SAVE MODEL
# ============================================

model.save('/kaggle/working/cifar_to_cinic_xception.keras')